# Gaussian maximum-likelihood selection and Guerrero smoothness

This chapter compares classical GCV with Gaussian marginal maximum likelihood (MLE) for **in-sample finite-difference smoothing**. It also demonstrates the two smoothness scales and the observed-difference mean mu="data". All data are simulated using a fixed random seed.

## What the likelihood assumes

For the n-d penalized spectral coordinates z_j with strictly positive penalty eigenvalues delta_j, we assume a latent Gaussian trend variance sigma²/(lambda delta_j) and observation noise variance sigma². Their marginal variance is sigma²[1 + 1/(lambda delta_j)].

We **profile sigma²**, remove the unpenalized polynomial/nullspace coordinates, and minimize twice the negative marginal log-likelihood up to a constant. Thus this is REML-style Gaussian marginal likelihood; it is not the negative of the least-squares objective. This is inspired by the Gaussian marginal likelihood of Biessy (2026, Section 5.3), with an additional profiled-variance step.

For mu="data", mu is the *arithmetic mean of the observed d-th differences*. We plug it into the likelihood; we do not claim joint maximum likelihood of mu and lambda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother, select_lambda

rng = np.random.default_rng(7)
t = np.arange(80)
truth = 1.5 + 0.025*t + 0.6*np.sin(t/15)
y = truth + rng.normal(scale=0.35, size=t.size)
solver = TrendSmoother(len(y), order=2)
mu_data = np.mean(solver.D @ y)
print("Mean observed second difference (mu='data'):", mu_data)

## Compare selection criteria

GCV uses the classical *in-sample* formula with fixed mu=0. MLE uses a Gaussian variance-component model and can take either fixed mu=0 or the empirical plug-in mu="data". These are different assumptions, so compare the resulting fitted trends rather than interpreting either score as a forecasting metric.

In [ ]:
gcv = select_lambda(solver, y, criterion="gcv", mu=0.0)
mle_zero = select_lambda(solver, y, criterion="mle", mu=0.0)
mle_data = select_lambda(solver, y, criterion="mle", mu="data")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, y, ".", alpha=0.4, label="Simulated noisy data")
ax.plot(t, truth, lw=2, label="Synthetic true trend")
for result, mean, label in [
    (gcv, 0.0, "GCV, mu=0"),
    (mle_zero, 0.0, "Gaussian MLE, mu=0"),
    (mle_data, "data", "Gaussian MLE, plug-in mu"),
]:
    trend = solver.fit(y, lambda_=result.lambda_, mu=mean)
    ax.plot(t, trend.trend, label=label)
    print(label, "lambda =", result.lambda_,
          "Guerrero smoothness =", trend.guerrero_smoothness)
ax.set(xlabel="Time", ylabel="Value", title="In-sample smoothing on synthetic data")
ax.legend(fontsize=8)
plt.show()

## Guerrero's original denominator

The library provides both S = (n - tr(H))/(n-d), normalized to [0,1], and Guerrero's **original** S_G = (n - tr(H))/n, whose upper limit is (n-d)/n. The fitted trend depends on lambda, not on the label used for smoothness. Both inverse mappings are supported.

In [ ]:
penalties = np.geomspace(1e-3, 1e5, 100)
original = [solver.smoothness(x, normalized=False) for x in penalties]
normalized = [solver.smoothness(x, normalized=True) for x in penalties]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(penalties, original, label="Guerrero original: denominator n")
ax.plot(penalties, normalized, label="Normalized: denominator n-d")
ax.axhline((solver.n - solver.order) / solver.n, ls=":", lw=1)
ax.set(xscale="log", xlabel="Penalty lambda", ylabel="Smoothness",
       title="Two valid conventions, one penalty parameter")
ax.legend()
plt.show()

target = solver.smoothness(mle_zero.lambda_, normalized=False)
recovered = solver.lambda_for_smoothness(target, normalized=False)
print("MLE lambda:", mle_zero.lambda_)
print("Recovered lambda from original smoothness:", recovered)
np.testing.assert_allclose(recovered, mle_zero.lambda_, rtol=1e-7)

## Estimating both Gaussian variance components

The penalty is the **ratio** of two variances, not itself a variance: lambda = sigma_epsilon² / sigma_eta². Here sigma_epsilon² is the independent **observation-noise variance**, while sigma_eta² is the variance of the **latent d-th differences** around mu. It is not the variance of the entire fitted trend.

After selecting lambda, the same profiled Gaussian likelihood yields sigma_epsilon² = sum(z_j²/c_j)/(n-d), where c_j = 1 + 1/(lambda delta_j), and sigma_eta² = sigma_epsilon²/lambda. These estimates are not the raw fitted-residual MSE. If lambda is infinite, sigma_eta² = 0. The data-derived mean option conditions on the plug-in mu; no joint MLE over mu is claimed.

In [ ]:
from trendestimation import estimate_variance_components

print("Profile variance components for numeric mu=0:")
estimates = estimate_variance_components(
    solver, y, lambda_=mle_zero.lambda_, mu=0.0
)
print("Observation variance:", estimates.observation_variance)
print("Latent d-th difference variance:", estimates.difference_variance)
print("Observation SD:", estimates.observation_sd)
print("Difference SD:", estimates.difference_sd)
print("Penalty/variance ratio:", estimates.lambda_)

# Same model, conditioned on the observed-difference plug-in mean:
plugin = estimate_variance_components(
    solver, y, lambda_=mle_data.lambda_, mu="data"
)
print("Plug-in mu:", plugin.mu)
print("Plug-in observation variance:", plugin.observation_variance)
print("Plug-in difference variance:", plugin.difference_variance)
if np.isfinite(plugin.lambda_):
    np.testing.assert_allclose(
        plugin.observation_variance / plugin.difference_variance,
        plugin.lambda_,
        rtol=1e-10
    )

## Interpretation and limits

- This library estimates **in-sample trends**, not future forecasts.
- lambda=0 is excluded from the Gaussian marginal likelihood: the latent prior becomes improper. The exact infinite-lambda limit is included.
- The two estimated variances are **observation noise** and **latent differences**, not trend variance or residual MSE.
- With difference order d=0, profiled variance components are not identifiable; MLE selection rejects this case.
- Classical CV/GCV/AICc/BIC require fixed numeric mu. Their leverage/EDF formulas are not exact leave-one-out refits of a mu estimated from the observed differences.
- mu="data" is a plug-in in the MLE criterion; uncertainty from estimating mu is not included.

References: Guerrero (2007); Cortés-Toto, Guerrero & Reyes (2017); Biessy (2026, Section 5.3).